# 4. Gaussian Process Regression

## Physics-informed GPR

Instead of training a Gaussian Process directly on resonance frequencies, we construct a Physics-Informed surrogate using discrepancy (residual) learning.
The baseline flexural resonance frequency for a thin square plate of half-side $a$, thickness $h$, Young's modulus $E$, density $\rho$, and Poisson's ratio $\nu$ is modeled as:

$$f_0 = \frac{1}{a^2} \sqrt{\frac{E h^2}{12 (1 - \nu^2) \rho}}$$

This will be our prior mean function $m(x)$, and the $\mathcal{GP}$ only learns the residual discrepancy $\delta(x)$ between simulations and the analytical model.
In standard residual modeling, the discrepancy is assumed to be purely additive:

$$f_0(\mathbf{x}) = m(\mathbf{x}) + \delta(\mathbf{x}), \quad \delta(\mathbf{x}) \sim \mathcal{GP}\left(0, k(\mathbf{x}, \mathbf{x}')\right)$$

However, we know that the simulation result is a scaled correction of the analytical formula:

$$f_0(\mathbf{x}) = m(\mathbf{x}) \cdot \delta_{\text{mult}}(\mathbf{x})$$

So we directly model the dimensionless scaling ratio:

$$\delta_{\text{mult}}(\mathbf{x}) = \frac{f_0(\mathbf{x})}{m(\mathbf{x})}$$

To train a standard zero-mean Gaussian Process, we subtract the nominal factor of $1$ so the residual centers around zero:$$\tilde{\delta}(\mathbf{x}) = \frac{f_0(\mathbf{x})}{m(\mathbf{x})} - 1, \quad \tilde{\delta}(\mathbf{x}) \sim \mathcal{GP}\left(0, k(\mathbf{x}, \mathbf{x}')\right)$$

At test time, the physical frequency is recovered as:

$$\hat{f}_0(\mathbf{x}) = m(\mathbf{x}) \cdot \left(1 + \hat{\tilde{\delta}}(\mathbf{x})\right)$$

**Goal is the same as in the previous examples:** Your task is to answer *How Low Can We Go?* and minimize the number of calls of computationally expensive simulation calls `get_freq()`. 
Since the surrogate inherits the scaling $f_0 ~ h / a^2$ a priori, it should be possible to reduce it quite significantly.





In [ ]:
# import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C

from gp_utils import get_freq

### Configuration and Parameter Bounds


In [ ]:
TARGET_FREQ = 45000.0  # target fundamental frequency in Hz (adjustable)
target_khz = TARGET_FREQ / 1e3

# Parameter bounds:
BOUNDS = np.array([
    [0.5e-3, 2.0e-3],    # a: (half-side, m)
    [5.0e-6, 20.0e-6],    # h: (thickness, m)
    [20.0e-6, 50.0e-6],  # p: (pitch, m)
    [2.0e-6, 10.0e-6]     # d: (diameter, m)
])

lower_b = BOUNDS[:, 0]
upper_b = BOUNDS[:, 1]

def to_unit(x):
    """Normalize physical parameters into [0, 1]."""
    return (x - lower_b) / (upper_b - lower_b)

def from_unit(u):
    """Unscale unit coordinates back to physical units."""
    return lower_b + u * (upper_b - lower_b)

def sample_valid_designs(n_samples, rng):
    # Samples n_init points uniformly across the parameter box [a, h, p, d] 
    # using random scaling between the lower and upper bounds (rng)

    u = rng.rand(n_samples, 4)
    x = from_unit(u)
    # Ensure R = (d/p)^2 < 0.5
    x[:, 3] = np.minimum(x[:, 3], x[:, 2] * np.sqrt(0.49))
    return x


In [ ]:
def mean_physics_informed(X_phys, E=160e9, nu=0.22, rho=2330.0, C=1.0):
    """
    Computes the physics-informed analytical prior mean frequency in kHz.
    
    Formula:
        f_prior = C * (1 / a^2) * sqrt( (E * h^2) / (12 * (1 - nu^2) * rho) )
    """
    X_phys = np.atleast_2d(X_phys)
    a = X_phys[:, 0]  # plate half-side (m)
    h = X_phys[:, 1]  # thickness (m)
        
    # Evaluate formula: (1 / a^2) * sqrt( (E * h^2) / (12*(1 - nu^2)*rho) )
    f_prior_hz = C * (h / (a**2)) * np.sqrt(E / (12.0 * (1.0 - nu**2) * rho))
    
    f_prior_khz = f_prior_hz / 1e3  # convert to kHz
    
    return f_prior_khz.squeeze()

### Initial Dataset Generation and GPR Setup


In [ ]:
# TODO: play around with these numbers:
N_TRAIN = 10
N_CANDIDATES = 50000

# Fix random seed for deterministic sampling and kernel optimization
rng = np.random.RandomState(42)

# ==============================================================================
# Initial Design Generation & Physics Evaluation
# ==============================================================================
# Generate initial N_TRAIN points within bounds
X_train_phys = sample_valid_designs(N_TRAIN, rng)

# Evaluate exact fundamental natural frequency f_0 (scaled to kHz for numerical stability)
y_train_khz = np.array([
    get_freq(row[0], row[1], row[2], row[3], m=1, n=1) / 1e3
    for row in X_train_phys
])


# ==============================================================================
# Gaussian Process Fit
# ==============================================================================

# Compute physical prior baseline on training data
m_train_khz = mean_physics_informed(X_train_phys)

# Compute the dimensionless relative ratio discrepancy: delta_tilde = (y / m) - 1
delta_train = (y_train_khz / m_train_khz) - 1.0

# Fit GP on dimensionless residuals (variance scaled around ~1.0)
kernel = C(1.0, (1e-3, 1e2)) * Matern(length_scale=[0.5, 0.5, 0.5, 0.5], nu=2.5)
gp = GaussianProcessRegressor(
    kernel=kernel, 
    alpha=1e-4, 
    n_restarts_optimizer=10, 
    random_state=42
)
gp.fit(to_unit(X_train_phys), delta_train)



# ==============================================================================
# Candidate Space & Surrogate Inversion
# ==============================================================================
# Generate a dense candidate pool
candidates_phys = sample_valid_designs(N_CANDIDATES, rng)
candidates_u = to_unit(candidates_phys)

# Predict candidates: Prior Mean + GP Discrepancy
m_cand_khz = mean_physics_informed(candidates_phys)
delta_pred, delta_std = gp.predict(candidates_u, return_std=True)
# Reconstruct predicted frequency: f_hat = m * (1 + delta)
pred_f_khz = m_cand_khz * (1.0 + delta_pred)
# Propagate uncertainty: std(f) = m * std(delta)
pred_std_khz = m_cand_khz * delta_std

# Select candidate closest to target frequency with low uncertainty
error_khz = np.abs(pred_f_khz - target_khz)

best_idx = np.argmin(error_khz + 0.5 * pred_std_khz)
best_params = candidates_phys[best_idx]
pred_target_khz = pred_f_khz[best_idx]
pred_target_std = pred_std_khz[best_idx]

# Validate against true physics
true_freq = get_freq(*best_params, m=1, n=1)
rel_error = abs(true_freq - TARGET_FREQ) / TARGET_FREQ * 100

print("Optimal Design Found:")
print(f"  Plate Half-side a:    {best_params[0]*1e6:.3f} um")
print(f"  Thickness h:          {best_params[1]*1e6:.2f} um")
print(f"  Pitch p:              {best_params[2]*1e6:.2f} um")
print(f"  Diameter d:           {best_params[3]*1e6:.2f} um")
print(f"  GP Predicted Freq:    {pred_target_khz:.2f} +/- {1.96*pred_target_std:.2f} kHz")
print(f"  Actual Physics Freq:  {true_freq/1e3:.2f} kHz (Target: {target_khz:.2f} kHz)")
print(f"  Absolute Error:       {rel_error:.3f}%")

Plot how the surrogate model looks:

In [ ]:
param_labels = ['a / mm', 'h / um', 'p / um', 'd / um']
param_names = ['Plate Half-side a', 'Thickness h', 'Pitch p', 'Diameter d']
scale_factors = [1e3, 1e6, 1e6, 1e6]

plt.figure(figsize=(10, 5))

for i in range(4):
    plt.subplot(2, 2, i + 1)
    
    sweep_vals = np.linspace(BOUNDS[i, 0], BOUNDS[i, 1], 250)
    X_slice_phys = np.tile(best_params, (250, 1))
    X_slice_phys[:, i] = sweep_vals
    
    # 1. Evaluate physical prior baseline along the 1D slice
    m_slice_khz = mean_physics_informed(X_slice_phys)
    
    # 2. Predict dimensionless ratio discrepancy via GP
    X_slice_u = to_unit(X_slice_phys)
    delta_pred, delta_std = gp.predict(X_slice_u, return_std=True)
    
    # 3. Total predicted mean = Prior Mean * (1 + GP Ratio Discrepancy)
    mu_khz = m_slice_khz * (1.0 + delta_pred)
    
    # 4. Propagate uncertainty: std(f) = m * std(delta)
    std_khz = m_slice_khz * delta_std
    
    # Ground truth frequency curve
    true_slice_khz = np.array([
        get_freq(row[0], row[1], row[2], row[3], m=1, n=1) / 1e3
        for row in X_slice_phys
    ])
    
    x_plot = sweep_vals * scale_factors[i]
    x_opt = best_params[i] * scale_factors[i]
    
    
    # Model predictions and true 1D physics slice
    plt.plot(x_plot, true_slice_khz, 'k--', linewidth=1.6, label='get_freq()')
    plt.plot(x_plot, mu_khz,  color='darkorange', linewidth=2.0, label='GP Posterior Mean')
    plt.fill_between(
        x_plot,
        mu_khz - 2 * std_khz,
        mu_khz + 2 * std_khz,
        color='orange',
        alpha=0.25,
        label=r"$\pm 2\sigma$"
    )


    plt.axhline(target_khz, color='crimson', linestyle=':', linewidth=1.5, label=f'Target ({target_khz:.1f} kHz)')
    plt.axvline(x_opt, color='darkgreen', linestyle='-', linewidth=2.0, label='Chosen Design')
    
    plt.title(f'Frequency Response vs. {param_names[i]}')
    plt.xlabel(param_labels[i])
    plt.ylabel('Fundamental Frequency / kHz')
    plt.grid(True, linestyle=':', alpha=0.6)
    
    if i == 0:
        plt.legend(loc='upper right', framealpha=0.85)

plt.tight_layout()
plt.show()